# Lab 6: Particle Filter

## 구현 (25점)

이 랩에서는 상태 추정을 위한 particle filter 를 구현한다. 상태가 $\bar{x} = [x,y]$ 로 주어지는 시스템을 생각하자. 여기서 $x$ 는 시스템의 x-위치, $y$ 는 y-위치다. 시스템의 동역학 모델은 완벽하게 안다고 가정한다:
$$x_{t+1} = x_t + 0.1 x_t,$$
$$y_{t+1} = y_t - 0.2 y_t.$$
상태가 원점으로부터 떨어진 거리를 (불완전하게) 추정해주는 센서가 있다고 하자. 충분한 실험 끝에 다음이 센서의 좋은 모델임을 알아냈다:
$$p(z_t | \bar{x}_t) = \mathcal{N}(\|\bar{x}_t\|, \sigma_\text{meas}^2),$$
여기서 $\sigma_\text{meas}^2$ 는 0.2 이다. 즉, 센서 측정값은 평균이 원점까지의 거리 $\|\bar{x}_t\|$ 이고 표준편차가 $\sigma_\text{meas}$ 인 가우시안 확률변수다.

또한 상태에 대한 초기 belief 는 평균이 $[0, 0]$ 이고 공분산이 $2 \times 2$ 항등행렬인 가우시안 분포, 즉 $\mathcal{N}(\bar{0}, I)$ 로 정의된다고 하자.

매 시점마다 시스템의 실제 상태 $\bar{x}_t$ 는 위 동역학에 따라 변한다. 하지만 우리는 상태를 완벽히 알지 못하므로, 받은 센서 측정값 $z_t$ 를 이용해 상태를 추정해야 한다. particle filter 를 사용해 상태에 대한 belief 를 (근사적으로) 유지하고 갱신할 것이다. 아래 코드에서 "TODO" 로 표시된 부분을 채워 particle filter 구현을 완성하는 것이 과제다. 먼저 라이브러리를 import 하고 동역학을 설정한다:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import time
import scipy.stats as st
from typing import List, Tuple
from IPython.display import HTML, Image

# NOTE: switched from '%matplotlib notebook' to '%matplotlib inline' for reliable rendering.
%matplotlib inline

################ Setup dynamics and sensor (Do not modify) ###########

def update_state(state: np.ndarray) -> np.ndarray:
    """
    Update the state using the (known) dynamics of the system.

    @param state: The current state of the system.

    @return: The updated state of the system.
    """
    state[0] = state[0] + 0.1 * state[0]
    state[1] = state[1] - 0.2 * state[1]
    return state

def get_sensor_reading(state_true: np.ndarray, sensor_std_dev: float) -> float:
    """
    Obtain a sensor measurement. This function simulates the sensor that the robot has.

    @param state_true: The true state of the robot.
    @param sensor_std_dev: The standard deviation of the sensor's noise.

    @return: A sensor measurement
    """
    z = np.random.normal(np.linalg.norm(state_true), sensor_std_dev)
    return z

아래에서 particle filter 를 구현한다. 구현이 실행되는 데 몇 초 걸릴 수 있는데, 이는 나중에 그림을 그리기 편하도록 (비효율적이지만) 모든 입자 정보를 전부 저장하기 때문이다. 실제로 particle filter 는 매우 효율적으로 동작한다. 다음 함수가 도움이 될 수 있다: [np.random.choice](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)

In [ ]:
def particle_filter(initial_state: np.ndarray, num_particles: int,
                    sensor_std_dev: float, init_mean: np.ndarray,
                    init_cov: np.ndarray, horizon: int) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    """
    시간 구간(horizon) 동안 particle filter 를 시뮬레이션한다.

    @param initial_state: 로봇의 실제 초기 상태 (2-벡터).
    @param num_particles: 필터의 입자 개수.
    @param sensor_std_dev: 센서 모델 출력의 표준편차.
    @param init_mean: 초기 분포의 평균 (2-벡터).
    @param init_cov: 초기 분포의 공분산 (2x2 행렬).
    @param horizon: 시뮬레이션할 스텝 수.

    @return: 2-튜플. 첫 번째는 실제 상태값의 히스토리, 두 번째는 입자값의 히스토리.
    """
    weights = np.zeros(num_particles)
    state_true = initial_state.copy()

    # 초기 belief N(init_mean, init_cov) 에서 샘플링하여 입자 집합을 초기화한다.
    # 이 샘플 집합이 bel(x_0) 에 대한 (근사) 표현이다.
    particles = np.random.multivariate_normal(init_mean, init_cov, num_particles)

    particle_history = [particles.copy()]
    state_true_history = [state_true.copy()]

    for t in range(horizon):  # 각 시각 t = 0, 1, 2, ... 에 대해

        ######### 수정 금지 #############
        # 실제 상태를 한 스텝 전진시키고, 노이즈가 섞인 센서 측정값(원점까지 거리)을 받는다.
        state_true = update_state(state_true)
        state_true_history.append(state_true.copy())

        z_t = get_sensor_reading(state_true, sensor_std_dev)
        #####################################

        ####### Particle filter 코드 ########
        for i in range(num_particles):  # 각 입자에 대해
            ##### TODO: Dynamics update (5점) #####
            # 예측: 각 입자를 (노이즈 없는) 동역학으로 한 스텝 민다.
            # 동역학에 프로세스 노이즈가 없으므로 결정론적이다(샘플링 불필요).
            # 이 루프가 끝나면 입자 집합은 예측 belief bel_bar(x_t) 를 근사한다.
            particles[i] = update_state(particles[i])

            # 중요도 가중치(importance weight): 이 입자가 측정값 z_t 를 얼마나 잘 설명하는지.
            # 센서 모델 p(z_t | x) = N( ||x|| , sensor_std_dev^2 ) 의 밀도를 z_t 에서 평가한다.
            weights[i] = st.norm(np.linalg.norm(particles[i]), sensor_std_dev).pdf(z_t)
            #####################################################

        ##### TODO: 재샘플링(resampling) 단계 구현 (15점) #################
        # 가중치를 확률분포로 정규화한다.
        # (전체 가중치로 나누는 것이 정규화 상수 p(z_t) 로 나누는 역할을 한다.)
        probs = weights / np.sum(weights)

        # 가중치에 비례하여 num_particles 개의 인덱스를 복원추출(replace=True)한다.
        # 가중치가 큰 입자는 여러 번 복제되고, 작은 입자는 사라지는 경향이 있다.
        # 이는 "likelihood 를 곱하고 정규화"하는 과정을 샘플로 구현한 것으로,
        # 재샘플링된 집합은 교정된 belief bel(x_t) 를 근사한다.
        idx = np.random.choice(num_particles, size=num_particles, replace=True, p=probs)
        particles = particles[idx]
        ###########################################################

        # 이후 그림/애니메이션을 위해 현재 입자 집합의 복사본을 저장한다.
        particle_history.append(particles.copy())

    return state_true_history, particle_history

이 셀이 실제로 필터를 실행한다.

In [ ]:
# Number of particles
num_particles = 1000

# Initialize particles using initial belief
init_mean = [0.0, 0.0]
init_cov = [[5.0, 0], [0, 5.0]]  # covariance is the identity matrix

# Initialize true state
state_true = np.array([1.0, 1.0])

# Define sensor noise standard deviation
sensor_std_dev = 0.2

# Time horizon
horizon = 15

state_true_history, particle_history = particle_filter(state_true, num_particles, sensor_std_dev, init_mean, init_cov, horizon)

마지막으로 이 셀이 결과를 애니메이션으로 만든다. 비디오 렌더링에서 에러가 나면 FFMPEG 를 설치해야 할 수 있다. 셀이 에러 없이 실행됐는데 애니메이션이 표시되지 않으면, 노트북이 저장된 디렉터리에서 anim.gif 파일을 확인하라. 일부 브라우저(마이크로소프트 Edge 포함)는 파일이 올바르게 생성·저장되었더라도 애니메이션을 제대로 표시하지 못할 수 있다.

In [ ]:
plt.ioff()
fig, ax = plt.subplots()

ax.set_xlim([-5, 5])
ax.set_ylim([-5, 5])
ax.set_aspect('equal')

particle_scatter = ax.scatter([], [])
state_scatter = ax.scatter([], [])

def animate(i):
    particle_scatter.set_offsets(particle_history[i])
    state_scatter.set_offsets(state_true_history[i])

# ani = animation.FuncAnimation(fig, animate, frames=len(particle_history))
import matplotlib
ani = matplotlib.animation.FuncAnimation(fig, animate, frames=len(particle_history))

ani.save('./anim.gif', writer='pillow', fps=100)
Image(url='./anim.gif')

## 논의 (5점)

particle filter 를 올바르게 구현했다면, 입자들이 두 곳에 뭉치는 것을 관찰할 수 있을 것이다. 아래 셀에 이 현상이 왜 일어나는지, 그리고 필터가 올바른 상태로 수렴하도록 시스템을 어떻게 바꿀 수 있을지 간단히 설명하라.

### 답안:

입자가 **두 곳**으로 뭉치는 이유는 센서가 **원점까지의 거리** $\|\bar{x}_t\|$ 만 측정하기 때문이다. 원점에서 같은 거리에 있는 두 상태는 동일한 측정값을 내므로 센서가 구별하지 못한다. 특히 거리만 재는 센서에게는 점 $(r, 0)$ 과 그 대칭점 $(-r, 0)$ 이 완전히 동일하게 보인다.

동역학이 이를 구체화한다: $y_{t+1} = 0.8\,y_t$ 가 $y \to 0$ 으로 수렴시키고 $x_{t+1} = 1.1\,x_t$ 가 $x$ 를 발산시키므로, 입자들이 x 축 근처로 모여 결국 대칭인 두 후보 $(+r, 0)$ 과 $(-r, 0)$ 만 남는다. 따라서 belief 는 **쌍봉(bimodal, 봉우리 두 개)** 이 되고, 이는 단일 가우시안(예: Kalman/EKF 의 belief)으로는 표현할 수 없지만 particle filter 는 표현할 수 있다.

필터가 올바른 상태로 수렴하게 하려면 **거리만으로 생기는 대칭을 깨야** 한다. 예를 들어:
- **서로 다른 기준점**까지의 거리를 재는 센서를 추가로 두 개 더한다.  세 개의 거리 측정이 있으면 (기준점들이 일직선상에 있지 않는 한) 세 원의 교점이 한 점으로 확정된다.
- 또는 **부호 / 위치 / 방향** 정보를 주는 센서를 추가한다 (예: $x$ 를 직접 측정하거나, 원점에 대한 방위각을 측정).

이 중 무엇이든 $(+r,0)/(-r,0)$ 모호성을 없애므로 belief 가 단봉이 되어 입자가 올바른 한 곳으로 수렴한다.